# 03 — Imputation Methods

We apply all five imputation methods to both datasets under all three missing mechanisms at 30% missing rate.

Methods:
1. Mean / Median / Mode (baseline)
2. KNN Imputation
3. MissForest
4. MICE
5. Autoencoder (complete_cases and mean_fill strategies)

Results are saved for evaluation in notebook 04.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import sys
sys.path.append('..')
from src.missing_generator import introduce_mcar, introduce_mar, introduce_mnar
from src.imputers import (
    impute_mean_median_mode,
    impute_knn,
    impute_missforest,
    impute_mice,
    impute_autoencoder,
)

sns.set_theme(style='whitegrid')
%matplotlib inline
SEED = 42

## 1. Load Datasets

In [ ]:
df_housing = pd.read_csv('../data/raw/housing.csv')
df_adult   = pd.read_csv('../data/raw/adult.csv')
print(f'Housing: {df_housing.shape} | Adult: {df_adult.shape}')

## 2. Train / Test Split

We split before introducing missing values so that imputation methods are trained on `train` only — preventing data leakage.

In [ ]:
housing_train, housing_test = train_test_split(df_housing, test_size=0.2, random_state=SEED)
adult_train, adult_test     = train_test_split(df_adult,   test_size=0.2, random_state=SEED)

housing_train = housing_train.reset_index(drop=True)
housing_test  = housing_test.reset_index(drop=True)
adult_train   = adult_train.reset_index(drop=True)
adult_test    = adult_test.reset_index(drop=True)

print(f'Housing — train: {housing_train.shape} | test: {housing_test.shape}')
print(f'Adult   — train: {adult_train.shape}   | test: {adult_test.shape}')

## 3. Helper: run all methods on a dataset

In [ ]:
METHODS = {
    'Mean/Median/Mode': lambda tr, te: impute_mean_median_mode(tr, te),
    'KNN':              lambda tr, te: impute_knn(tr, te),
    'MissForest':       lambda tr, te: impute_missforest(tr, te, seed=SEED),
    'MICE':             lambda tr, te: impute_mice(tr, te, seed=SEED),
    'Autoencoder (complete_cases)': lambda tr, te: impute_autoencoder(tr, te, seed=SEED, training_strategy='complete_cases'),
    'Autoencoder (mean_fill)':      lambda tr, te: impute_autoencoder(tr, te, seed=SEED, training_strategy='mean_fill'),
}

def run_all_methods(train_missing, test_missing):
    results = {}
    for name, fn in METHODS.items():
        print(f'  Running {name}...')
        results[name] = fn(train_missing, test_missing)
    return results

## 4. California Housing

In [ ]:
housing_target_cols = ['MedInc', 'HouseAge', 'AveRooms']

housing_missing = {
    'MCAR': introduce_mcar(housing_test, columns=housing_target_cols, missing_rate=0.3),
    'MAR':  introduce_mar(housing_test,  target_col='MedInc', condition_col='HouseAge', missing_rate=0.3),
    'MNAR': introduce_mnar(housing_test, target_col='MedInc', missing_rate=0.3, direction='high'),
}

housing_train_missing = {
    'MCAR': introduce_mcar(housing_train, columns=housing_target_cols, missing_rate=0.3),
    'MAR':  introduce_mar(housing_train,  target_col='MedInc', condition_col='HouseAge', missing_rate=0.3),
    'MNAR': introduce_mnar(housing_train, target_col='MedInc', missing_rate=0.3, direction='high'),
}

housing_imputed = {}
for mechanism in ['MCAR', 'MAR', 'MNAR']:
    print(f'\nHousing — {mechanism}')
    housing_imputed[mechanism] = run_all_methods(
        housing_train_missing[mechanism],
        housing_missing[mechanism]
    )

## 5. Adult Census

In [ ]:
adult_target_cols = ['age', 'hours_per_week', 'capital_gain']

adult_missing = {
    'MCAR': introduce_mcar(adult_test, columns=adult_target_cols, missing_rate=0.3),
    'MAR':  introduce_mar(adult_test,  target_col='hours_per_week', condition_col='age', missing_rate=0.3),
    'MNAR': introduce_mnar(adult_test, target_col='capital_gain', missing_rate=0.3, direction='high'),
}

adult_train_missing = {
    'MCAR': introduce_mcar(adult_train, columns=adult_target_cols, missing_rate=0.3),
    'MAR':  introduce_mar(adult_train,  target_col='hours_per_week', condition_col='age', missing_rate=0.3),
    'MNAR': introduce_mnar(adult_train, target_col='capital_gain', missing_rate=0.3, direction='high'),
}

adult_imputed = {}
for mechanism in ['MCAR', 'MAR', 'MNAR']:
    print(f'\nAdult — {mechanism}')
    adult_imputed[mechanism] = run_all_methods(
        adult_train_missing[mechanism],
        adult_missing[mechanism]
    )

## 6. Distribution Comparison — Quick Visual Check

In [ ]:
col = 'MedInc'
mechanism = 'MCAR'

fig, ax = plt.subplots(figsize=(12, 5))
sns.kdeplot(housing_test[col], ax=ax, label='Original', linewidth=2, color='black')
for method_name, df_imp in housing_imputed[mechanism].items():
    sns.kdeplot(df_imp[col], ax=ax, label=method_name, linestyle='--')
ax.set_title(f'Distribution comparison — {col} under {mechanism} (30%)')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Save Imputed Datasets

In [ ]:
import pickle, os
os.makedirs('../data/processed', exist_ok=True)

with open('../data/processed/housing_imputed.pkl', 'wb') as f:
    pickle.dump(housing_imputed, f)

with open('../data/processed/adult_imputed.pkl', 'wb') as f:
    pickle.dump(adult_imputed, f)

# also save the original test sets and missing masks
housing_test.to_csv('../data/processed/housing_test_original.csv', index=False)
adult_test.to_csv('../data/processed/adult_test_original.csv', index=False)

print('Saved.')